# VoxFlow cleanup model
This notebook builds a synthetic noisy-to-clean dataset and trains a small text-cleanup model using T5/Flan-T5 style fine-tuning.


In [ ]:
# Install dependencies
# %pip install -q datasets transformers peft accelerate sentencepiece

import random
import re
from datasets import Dataset
from transformers import T5ForConditionalGeneration, T5Tokenizer, Seq2SeqTrainingArguments, Seq2SeqTrainer
from peft import LoraConfig, get_peft_model

MODEL_NAME = "google/flan-t5-small"

clean_sentences = [
    "Please review the update before we ship it.",
    "I will meet you at the office tomorrow morning.",
    "The project is moving along very well.",
    "Can you send the report before noon?",
]


def make_noisy(text: str) -> str:
    text = text.replace("Please", "Uh please")
    text = text.replace("I will", "I will actually")
    text = text.replace("The project", "The project uh")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

rows = [{"input": make_noisy(s), "output": s} for s in clean_sentences]
raw_dataset = Dataset.from_list(rows)
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)


def prepare_example(example):
    source = tokenizer(example["input"], truncation=True, max_length=128, padding="max_length", return_tensors="pt")
    target = tokenizer(example["output"], truncation=True, max_length=128, padding="max_length", return_tensors="pt")
    return {"input_ids": source["input_ids"][0], "attention_mask": source["attention_mask"][0], "labels": target["input_ids"][0]}

prepared = raw_dataset.map(prepare_example, remove_columns=raw_dataset.column_names)

model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
lora_config = LoraConfig(r=8, lora_alpha=16, target_modules=["q", "v"], lora_dropout=0.05, bias="none")
model = get_peft_model(model, lora_config)

training_args = Seq2SeqTrainingArguments(
    output_dir="./cleanup_demo_out",
    per_device_train_batch_size=1,
    learning_rate=1e-4,
    num_train_epochs=2,
    logging_steps=5,
    save_strategy="epoch",
)

trainer = Seq2SeqTrainer(model=model, args=training_args, train_dataset=prepared)
trainer.train()
print("Cleanup fine-tuning demo completed")
